# Train a Recommender - scikit-learn

This notebook uses the [Singular Value Decomposition (SVD)](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#) algorithm in the scikit-learn.surprise package to build a recommender model for movie ratings using [matrix factorization](https://en.wikipedia.org/wiki/Matrix_factorization_(recommender_systems)). The factor matrices that make up the fitted model are then saved as Snowflake tables, to allow deployment of the model as a simple Snowflake UDF.

Steps in this notebook:
1. Setup
2. Load and Inpect Ratings Data
3. Prepare Data for Training
4. Train and Evaluate Recommender Model
5. Transfer Model Parameters to Snowflake

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.functions import *
from snowflake.snowpark.session import Session

import pandas as pd
import numpy as np

# CONFIG_DIR = '/home/jovyan/.ssh'
CONFIG_DIR = '/Users/richardkirk/.ssh'
CONFIGFILE = CONFIG_DIR + '/sf_config'

## 1. Setup

#### Connect to Snowflake

In [ ]:
# Load configuration file
with open(CONFIGFILE) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

#### Install the Python package [Surprise](https://surpriselib.com/)

**!! MyNotes: I'm not using JupyterLab server, so installed with pip install !!**

This package--included in [Snowflake's Anaconda channel](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-packages#using-third-party-packages-from-anaconda)--contains classes for building and analyzing a recommender. 

**NOTE**: 
The cell below installs the package in the local 'snowpark' environment. Alternatively, you can perform the same install in a separate terminal window with code:

```
conda activate snowpark
conda install scikit-surprise -y -c https://repo.anaconda.com/pkgs/snowflake
```

You need to run this code only once in your JupyterLab server, so you may choose to avoid re-runs by changing the cell from a Code sell to a Raw cell after the first run. 

In [ ]:

# As not using JupyterLab server, I installed reguarlar way:
# pip install scikit-surprise

# %%bash 
# source /opt/conda/etc/profile.d/conda.sh
# conda activate ds_env
# conda install scikit-surprise -y -c https://repo.anaconda.com/pkgs/snowflake

## 2. Load and Inspect Ratings Data
#### Define a Snowpark DataFrame on the Snowflake table.

In [ ]:
movie_ratings = session.table('data_science_db.movielens.ratings')

#### Inspect the DataFrame

In [ ]:
movie_ratings.schema.fields

In [ ]:
movie_ratings.count()

In [ ]:
movie_ratings.show(4)

## 3. Prepare Data for Training
#### Discard 'unixtime' column.

In [ ]:
ratings = movie_ratings.drop('unixtime')

#### Map (**user**, **item**) labels

Provide mapping from input **userid** and **movieid** columns to integer sequences (0,1,2,3,...,n-1).

Look at the last few values of **movieid** in the ratings data:

In [ ]:
(ratings.
 select('movieid').
 distinct().
 sort(col('movieid').desc()).
 show(4))

These values are not contiguous. However, the SVD algorithm used here requires (**user**, **item**) matrix indexes as contiguous integers counting up from zero. Create a pair of DataFrames that map the original **userid** and **movieid** columns to these indexes.

Start with a mapping for **movieid**:

In [ ]:
from snowflake.snowpark.window import Window
item_mapping = (
    ratings.
    select('movieid').
    distinct().
    withColumn('item', row_number().over(Window.order_by(col('movieid'))) - 1).
    select ('movieid', 'item')
)

Spot check this mapping.

In [ ]:
# How many distinct values for movieid?
(ratings.
 select('movieid').
 distinct().
 count()
)

In [ ]:
# Lowest four movieids
item_mapping.sort(col('movieid').asc()).show(4)

In [ ]:
# Highest four movieids
item_mapping.sort(col('movieid').desc()).show(4)

Create similar mapping for **userid**.

In [ ]:
user_mapping = (
    ratings.
    select('userid').
    distinct().
    withColumn('user', row_number().over(Window.order_by(col('userid'))) - 1).
    select ('userid', 'user')
)

#### Load data for training

Map the original labels for (**userid**, **movieid**) to (**user**, **item**) and load as pandas DataFrame for training.

In [ ]:
trainingPDF = (ratings.
               join(user_mapping, 'userid', 'inner').
               join(item_mapping, 'movieid', 'inner').
               select('user', 'item', 'rating').
               sort('user', 'item').
               toPandas()
              )

## 4. Train and Evaluate Recommender Model
#### Declare model

In [ ]:
from surprise import SVD
recommender = SVD(n_factors=10, biased=False)

Note this declaration chooses hyperparameter setting **biased=False** in order to produce a simpler model for illustration. A setting of **biased=True** produces better model performance, and is preferable for production deployment (more on this below).

#### Cast data as needed for the algorithm.

In [ ]:
from surprise import Reader
from surprise import Dataset

reader = Reader(rating_scale = (0.5,5)) 
data = Dataset.load_from_df(trainingPDF[["USER", "ITEM", "RATING"]], reader)

#### Create train/test split; train model; make test predictions

In [ ]:
# 80/20 train/test split
from surprise.model_selection import train_test_split
trainset, testset = train_test_split(data, test_size=.2)

# Train model
recommender.fit(trainset)

# Provide predictions on test data
predictions = recommender.test(testset)

#### View predictions

**r_ui** is the actual rating in the test data; **est** is the predicted rating.

In [ ]:
predictions[0:4]

#### Measure model performance

In [ ]:
from surprise import accuracy
accuracy.rmse(predictions)

(Aside: A run of this algorithm with setting **biased=True** yields an RMSE of 0.8632, an improvement over this model. That fit includes two additional matrices for user bias and item bias for every user and item. To get the improved prediction, you take the prediction for any (user, item) pair (vector product of user factors and item factors) and add the bias for that user and the bias for that item. This notebook uses the simpler algorithm to produce only the two factor matrices, for simplicity of the deployment example.)

## 5. Transfer Model Parameters to Snowflake

#### Retrain model with all available data

In [ ]:
recommender.fit(data.build_full_trainset())

#### Prepare model parameters for saving to Snowflake

The factored matrices from the model are:
- **recommender.pu** - the user factors matrix
- **recommender.qi** - the item factors matrix

Convert these to suitable Snowpark DataFrames for saving to Snowflake tables.

In [ ]:
userFactorsList = recommender.pu.tolist()  # for Snowpark create_dataframe() call

userFactors = (
    session.create_dataframe(userFactorsList).  # Snowpark DataFrame
    selectExpr('array_construct(*) as features').  # Collect columns into an array (used as a vector)
    withColumn('user', row_number().over(Window.order_by(lit(1))) - 1). # Get row numbers
    select('user', 'features').
    join(user_mapping, 'user', 'inner'). # Map back to original userid
    withColumnRenamed('userid', 'id').
    select('id', 'features')
)

itemFactorsList = recommender.qi.tolist()  # for Snowpark create_dataframe() call

itemFactors = (
    session.create_dataframe(itemFactorsList).  # Similar to userFactors above
    selectExpr('array_construct(*) as features').
    withColumn('item', row_number().over(Window.order_by(lit(1))) - 1).
    select('item', 'features').
    join(item_mapping, 'item', 'inner').
    withColumnRenamed('movieid', 'id').
    select('id', 'features')
)

#### Save tables to Snowflake

In [ ]:
session.sql("create schema if not exists recommender").show()
session.sql("use schema recommender")
userFactors.write.mode("overwrite").save_as_table('recommender.user_factors')
itemFactors.write.mode("overwrite").save_as_table('recommender.item_factors')

### Snowpark vs SQL

Now if we want to stay in Snowpark, we have to use a Stored Procedure

Because the only type of UDF that can do a SELECT is a SQL UDF.  All others (Python, Java , JS, Scala) can NOT do any DDL, DML, or SQL with your SnowFlake account. 

Only Stored Procedures can do all three.

Below we show you how to create such a stored procedure. However, to easily use it on table, we will go back to Snowsight and create a UDF there.


In [ ]:
from snowflake.snowpark.types import DoubleType

curr_db = session.get_current_database()
curr_schema = session.get_current_schema()

def predict_rating(session:Session, userid, itemid) -> float:
    user_df = session.table("user_factors")
    item_df = session.table("item_factors")

    user_df = (user_df
        .filter(col('id') == userid) 
        .join_table_function(
            "flatten"
            ,col("FEATURES") # input - The column containing data to flatten
        )
        .select('index', col('value').alias('user_value'))
      )


    item_df = (item_df
        .filter(col('id') == itemid) 
        .join_table_function(
            "flatten"
            ,col("FEATURES") # input - The column containing data to flatten
        )
        .select('index', col('value').alias('item_value'))
      )

    joined_df = user_df.join(item_df, "INDEX") 
    return joined_df.select(round(sum(col('user_value') * col('item_value')), 2).alias('prediction')).collect()[0][0]

    
    
prediction_udf = (sproc(                           
         func = predict_rating
        ,return_type = DoubleType()
        ,input_types = [DoubleType(),DoubleType()]
        ,is_permanent = True       
        ,stage_location = '@~'      
        ,replace = True             
        ,session = session
        ,packages = ["snowflake-snowpark-python"]  # Package dependencies
        ,name = [curr_db, curr_schema, "PREDICTION"] # DB, Schema, and SQL function name
     )            
) 

In [ ]:
# Test our Python Stored Procedure
session.call("PREDICTION",1, 110)